In [1]:
import os
import re
import pandas as pd
from pathlib import Path

project_dir = Path.cwd().parent

In [2]:
data = pd.read_csv(os.path.join(project_dir, "data/test_results/Qwen2.5-14B-Instruct-Exercise-default-bs4-trainset_test_result.csv"))

In [6]:
data["format_prediction"] = data["prediction"].apply(lambda x: re.findall("<exercise>([\s\S]+?)(?:</exercise>|$)", x)[0])

In [5]:
for i in range(len(data)):
    sample = data.iloc[i]['prediction']
    # Try matching both cases: with and without </exercise>
    match = re.findall("<exercise>([\s\S]+?)(?:</exercise>|$)", sample)
    if len(match) == 0:
        print(sample)
        print("----")
        break

In [7]:
data.head()

,Input,Output,prediction,format_prediction
0,Give me a true or false statements exercise fo...,Here are some statements. Write true if the st...,"<|im_start|>system\nYou are Qwen, created by A...",Write true or false next to each statement bel...
1,Give me a combination of a group activity and ...,Work in a group to devise an internet safety c...,"<|im_start|>system\nYou are Qwen, created by A...",Work in a group to create a poster or other ca...
2,Give me a combination of a sequence completion...,\begin{enumerate}[label=\alph*.]\n\item Write ...,"<|im_start|>system\nYou are Qwen, created by A...",Complete these sequences.\n\n\begin{enumerate}...
3,"Give me a ""always sometimes or never true"" exe...","Is it always, sometimes or never true that the...","<|im_start|>system\nYou are Qwen, created by A...",True or false? Explain your answers.\n\begin{e...
4,Give me a fill-in-the-blanks exercise for a st...,Fill the spaces with the correct modal verbs.\...,"<|im_start|>system\nYou are Qwen, created by A...",Fill in the spaces with the correct form of th...


In [8]:
data.to_csv(os.path.join(project_dir, "data/test_results/Qwen2.5-14B-test_result.csv"), index=False)

In [ ]:
data = pd.read_csv(os.path.join(project_dir, "data/instruction_sets/instruct_test.csv"))

In [108]:
len(data)

860

In [2]:
log_text = open(os.path.join(project_dir, "sbatch_logs/test_all-15782428_trained_with_excercise_tag_+system_prompt_topp0.9_topk50_temperature0.7.out")).read()

In [101]:
def get_non_overlapping(text1, text2):
    # Check if text2 starts with text1
    if text2.startswith(text1):
        return text2[len(text1):]
    else:
        return text2

result = []

n = 860
for i in range(n):
    prompt = re.findall(f"<<{i}>> ([\s\S]+)\n<<{i}>>", log_text)[0]
    result1 = re.findall(f"<<{i}>> ([\s\S]+\n<<{i}>> )", log_text)[0]
    if i != n-1:
        result2 = re.findall(f"<<{i}>> ([\s\S]+?)\n<<{i+1}>>", log_text)[0]
    else:
        result2 = re.findall(f"<<{i}>> ([\s\S]+?)\n\n#########################", log_text)[0]
    result.append({"prompt": prompt, "prediction": get_non_overlapping(result1, result2)})

In [102]:
len(result)

860

In [ ]:
# pd.DataFrame(result).to_csv(os.path.join(project_dir, "data", "test_output_e2_gas4_lr2e-4_r8_a16.csv"), index=False)

In [109]:
result = pd.read_csv(os.path.join(project_dir, "data", "test_output_e2_gas4_lr2e-4_r8_a16.csv"))

In [110]:
result.head()

,prompt,prediction
0,Give me a true or false statements exercise fo...,Here are some statements about odd and even nu...
1,Give me a combination of a group activity and ...,Work in a group to prepare a campaign to infor...
2,Give me a combination of a sequence completion...,"Given the sequence: 2, 6, 12, 20, 30,..., what..."
3,"Give me a ""always sometimes or never true"" exe...","Is this statement always, sometimes or never t..."
4,Give me a fill-in-the-blanks exercise for a st...,Insert the correct form of the modal verbs int...


In [112]:
data.head()

,Input,Output
0,Give me a true or false statements exercise fo...,Here are some statements. Write true if the st...
1,Give me a combination of a group activity and ...,Work in a group to devise an internet safety c...
2,Give me a combination of a sequence completion...,\begin{enumerate}[label=\alph*.]\n\item Write ...
3,"Give me a ""always sometimes or never true"" exe...","Is it always, sometimes or never true that the..."
4,Give me a fill-in-the-blanks exercise for a st...,Fill the spaces with the correct modal verbs.\...


In [115]:
result['reference'] = data.Output

In [116]:
result

,prompt,prediction,reference
0,Give me a true or false statements exercise fo...,Here are some statements about odd and even nu...,Here are some statements. Write true if the st...
1,Give me a combination of a group activity and ...,Work in a group to prepare a campaign to infor...,Work in a group to devise an internet safety c...
2,Give me a combination of a sequence completion...,"Given the sequence: 2, 6, 12, 20, 30,..., what...",\begin{enumerate}[label=\alph*.]\n\item Write ...
3,"Give me a ""always sometimes or never true"" exe...","Is this statement always, sometimes or never t...","Is it always, sometimes or never true that the..."
4,Give me a fill-in-the-blanks exercise for a st...,Insert the correct form of the modal verbs int...,Fill the spaces with the correct modal verbs.\...
...,...,...,...
855,Give me a combination of a matching exercise a...,Match the clocks to the correct times and colo...,Draw lines to match the ordinal number with th...
856,Give me a combination of an illustration exerc...,Draw three different ways to divide a chocolat...,Draw three different ways to divide a chocolat...
857,Give me a combination of a descriptive exercis...,Describe an object that is long. Describe an o...,"Zero is an even number, but you cannot put 0 c..."
858,Give me a combination of a number comparison e...,Compare each pair of numbers. Fill in the boxe...,Compare these numbers and complete the sentenc...


# Version2

In [3]:
prompts = re.findall("<<\d+ Prompt>> ([\s\S]+?) <<Prompt>>", log_text)
generations = re.findall("<<\d+ Generation>> ([\s\S]+?) <<Generation>>", log_text)
len(prompts), len(generations)

(860, 860)

In [4]:
for one in generations:
    if not one.startswith("<exercise>"):
        print(1)
    if not one.endswith("</exercise>"):
        print("--------------------------------")
        print(one)
        print()

--------------------------------
<exercise>Write the name of each shape in the space provided.

\begin{enumerate}[label=\alph*.]
\item A \_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\

--------------------------------
<exercise>\begin{enumerate}[label=(\alph*)]
\item You are going to write a short story about a character who is trapped somewhere. You need to decide where the character is trapped.\\
\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\

In [22]:
generations = [one.replace("<exercise>", "").replace("</exercise>", "") for one in generations]

In [ ]:
test_data = pd.read_csv(os.path.join(project_dir, "data/instruction_sets/instruct_test.csv"))
references = test_data.Output

In [21]:
from lexicality.lexicality_eval import lexicality
for prediction_text, reference_text in zip(generations, references):
    # print(prediction_text, reference_text)
    result = lexicality(prediction_text, reference_text)
    print(result)
    break

{'FRE_score': 71.29178571428572, 'ROUGE-1_F1': 0.3275862068965517, 'ROUGE-S_F1': 0.08771929824561403, 'avg_sentence_length': 10.5, 'avg_sentence_count': 9, 'punctuation_distribution': {'period_ratio': 1.0, 'exclamation_ratio': 0.0, 'question_ratio': 0.0}}


In [22]:
pd.DataFrame({"prompt": prompts, "prediction": generations}).head()

,prompt,prediction
0,Give me a true or false statements exercise fo...,Which of these statements is true? Circle it.\...
1,Which of these statements is true? Circle it.\...,Work in a group to develop a campaign to promo...
2,Work in a group to develop a campaign to promo...,Complete the following sequence:\n\n\[\n\begin...
3,Complete the following sequence:\n\n\[\n\begin...,"Decide whether these statements are always, so..."
4,"Decide whether these statements are always, so...",Insert the correct form of the modal verbs int...


In [32]:
pd.DataFrame({"prompt": prompts, "prediction": generations}).to_csv(os.path.join(project_dir, "data/test_output_e2_gas4_lr2e-4_r8_a16.csv"), index=False)

# Convert dataset to json format

In [10]:
import json

with open(os.path.join(project_dir, "data", "instructions.json")) as f:
    train_data = json.load(f)

In [11]:
len(train_data)

8312

In [ ]:
instruct_train = pd.read_csv(os.path.join(project_dir, "data/instruction_sets/instructions.csv"))

In [13]:
len(instruct_train)

8312

In [14]:
instruct_train_json = [{"messages": [{"role": "user", 
                                      "content": prompt},
                                    {"role": "assistant",
                                     "content": f"<exercise>{groundtruth}</exercise>"}]}
                        for prompt, groundtruth in zip(instruct_train.Input, instruct_train.Output)]

In [15]:
with open(os.path.join(project_dir, "data", "instruct_chatml_with_tag.json"), 'w') as f:
    json.dump(instruct_train_json, f)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=os.path.join(project_dir, "data/instruction_sets/instruct_chatml_with_tag.json"), split="train", num_proc=4)

In [8]:
dataset[0]['messages']

[{'content': 'Give me a combination of a practical exercise and an activity for a 2nd grade student learning computer science.',
  'role': 'user'},
 {'content': "<exercise>Create a document file called 'My hobby' to use as a poster and add an image to it.\nChange the size of the image in your poster.  \nMove the image to the middle of the page.  \nSave your file.  \n\n\\paragraph{Extra challenge}  \nFind a second image to add to your poster. Place the two images on your poster.  \n\n\\paragraph{Think again}\nWould you prefer to add one large image or many small images to your poster? Give reasons for your choice.</exercise>",
  'role': 'assistant'}]